# Natural Language Processing with Attention Models

Если sequence models научили NLP работать с контекстом как с последовательной памятью, то attention models сделали следующий, ещё более радикальный шаг:

**они отказались от идеи, что весь контекст нужно протаскивать через рекуррентное скрытое состояние, и вместо этого научились напрямую выбирать, на какие элементы входа смотреть при вычислении представления каждого токена.**

В субтитрах курса это формулируется очень ясно: RNN, LSTM и GRU частично помогают, но на длинных последовательностях всё равно страдают от bottleneck и потери информации; attention вводится как способ решить именно эти проблемы, а transformers идут ещё дальше и строятся только на attention-механизмах, без рекуррентных сетей.

---

# 1. Место attention models в общей карте знаний NLP

```text
NLP
├── Text processing and vector spaces
│   ├── preprocessing
│   ├── Bag of Words / TF-IDF
│   ├── Logistic Regression
│   └── Naive Bayes
├── Probabilistic models
│   ├── n-grams
│   ├── smoothing
│   ├── HMM
│   └── Viterbi
├── Sequence models
│   ├── neural language models
│   ├── RNN
│   ├── LSTM / GRU
│   ├── bidirectional models
│   └── seq2seq
└── Attention models
    ├── attention mechanism
    ├── encoder-decoder attention
    ├── self-attention
    ├── masked self-attention
    ├── multi-head attention
    ├── transformer
    ├── encoder-only models
    ├── decoder-only models
    ├── encoder-decoder transformers
    ├── pretraining and transfer learning
    └── modern LLMs
```

## Что здесь фундаментально

Фундаментальные темы раздела:

- attention as differentiable retrieval;
- queries, keys, values;
- alignment;
- self-attention;
- masking;
- multi-head attention;
- positional encoding.

## Что здесь производно

Производные темы:

- transformer encoder;
- transformer decoder;
- BERT;
- GPT;
- T5/BART;
- modern pretraining;
- multitask text-to-text learning.

Этот раздел завершает классическую эволюцию NLP:

**BoW → probabilistic models → RNN/LSTM → attention → transformers → LLM**.

---

# 2. Онтология ключевых терминов раздела

### Attention
- **тип:** neural mechanism
- **определение:** механизм, который вычисляет, какие части входа наиболее релевантны для текущего вычисления.
- **связи:** alignment, seq2seq, transformers.
- **роль:** снимает bottleneck фиксированного контекстного вектора.

### Query / Key / Value
- **тип:** learned representations
- **определение:** три матричных представления, через которые attention вычисляет релевантность и контекст.
- **связи:** scaled dot-product attention, self-attention, multi-head attention.
- **роль:** формализуют «что я ищу», «где искать» и «что извлечь».

### Self-Attention
- **тип:** attention mechanism
- **определение:** attention, где queries, keys и values приходят из одной и той же последовательности.
- **связи:** transformer encoder, contextual representations.
- **роль:** позволяет каждому токену учитывать все остальные токены.

### Masked Self-Attention
- **тип:** causal attention mechanism
- **определение:** self-attention с запретом смотреть в будущие позиции.
- **связи:** decoder-only transformers, autoregressive language modeling.
- **роль:** обеспечивает корректную генерацию слева направо.

### Multi-Head Attention
- **тип:** parallel attention mechanism
- **определение:** набор параллельных attention heads с разными линейными проекциями.
- **связи:** transformer, Q/K/V projections.
- **роль:** позволяет модели одновременно учить разные отношения между токенами.

### Positional Encoding
- **тип:** sequence representation component
- **определение:** способ добавить информацию о порядке токенов в архитектуру без рекуррентности.
- **связи:** transformer, token order.
- **роль:** компенсирует отсутствие RNN.

### Transformer
- **тип:** neural architecture family
- **определение:** архитектура, основанная на self-attention, feed-forward layers, residual connections и normalization.
- **связи:** BERT, GPT, T5, LLMs.
- **роль:** стандарт современного NLP.

### Encoder-only model
- **тип:** transformer architecture variant
- **определение:** модель, состоящая только из encoder stack.
- **связи:** BERT, representation learning, classification, NER.

### Decoder-only model
- **тип:** transformer architecture variant
- **определение:** модель, состоящая только из decoder stack с causal masking.
- **связи:** GPT, text generation, causal language modeling.

### Encoder-Decoder Transformer
- **тип:** transformer architecture variant
- **определение:** модель с encoder и decoder stack.
- **связи:** T5, BART, translation, summarization.

### MLM (Masked Language Modeling)
- **тип:** pretraining objective
- **определение:** задача предсказания скрытых токенов по двустороннему контексту.
- **связи:** BERT, encoder models.

### Causal Language Modeling
- **тип:** pretraining objective
- **определение:** предсказание следующего токена по предыдущим.
- **связи:** GPT, decoder-only transformers.

---

# 3. Почему attention вообще понадобился

## 3.1. Интуитивная идея

В классическом seq2seq encoder должен был сжать всё входное предложение в один вектор. Для коротких предложений это работало терпимо, но для длинных последовательностей возникал сильный information bottleneck.

Курс прямо говорит, что LSTM и GRU помогают лишь частично: при длинных последовательностях сохраняются loss of information и vanishing gradient issues. Включение attention описывается как способ решить эти проблемы, а transformers как архитектура, которая вообще больше не требует recurrent networks.

## 3.2. Что делает attention концептуально

Attention заменяет идею:

**“сжать всё в один вектор”**

на идею:

**“на каждом шаге смотреть на релевантные части входа и вычислять контекст динамически”**.

Это критический сдвиг. Теперь decoder при генерации каждого нового слова может по-разному обращаться к входному предложению, а encoder может строить представление токена, учитывая все другие токены.

## 3.3. Исторический контекст

Сначала attention появился как дополнение к seq2seq в machine translation. Затем self-attention стал ядром transformer architecture. После этого transformers стали стандартом практически для всего современного NLP, включая BERT, GPT и T5. Курс прямо связывает transformer с революцией в NLP и называет его стандартом для современных large language models.

---

# 4. Attention как механизм выравнивания и извлечения контекста

## 4.1. Интуитивная идея

В machine translation decoder на шаге генерации очередного слова должен понять, какие слова входного предложения сейчас важны. Это и есть задача alignment.

Курс подчёркивает, что attention хорошо работает даже для языков с разным порядком слов: веса attention выравнивают семантически соответствующие фрагменты независимо от позиции в предложении.

## 4.2. Формальная постановка

Пусть есть query $q$, набор keys $k_1,\dots,k_n$ и corresponding values $v_1,\dots,v_n$.

Тогда attention вычисляет веса релевантности:

$$
\alpha_i = \text{softmax}(score(q, k_i))
$$

и затем формирует контекстный вектор:

$$
c = \sum_{i=1}^{n} \alpha_i v_i
$$

Интуитивно:

- **query** — что мы сейчас ищем;
- **key** — по чему сравниваем;
- **value** — что извлекаем.

## 4.3. Почему это похоже на retrieval

Attention можно понимать как мягкий, дифференцируемый поиск по памяти. Курс прямо связывает attention с идеями information retrieval. Вместо жёсткого выбора одного элемента модель распределяет веса по всем кандидатам.

---

# 5. Scaled Dot-Product Attention

## 5.1. Интуитивная идея

Самый важный и практически стандартный вариант attention в transformers — это scaled dot-product attention.

Курс называет его “heart and soul of transformers” и подчёркивает, что он состоит всего из матричных умножений и SoftMax, поэтому отлично подходит для GPU/TPU и массовой параллелизации.

## 5.2. Формула

Пусть:

- $Q$ — матрица queries;
- $K$ — матрица keys;
- $V$ — матрица values.

Тогда:

$$
\text{Attention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

где $d_k$ — размерность key vectors.

Курс буквально описывает эти шаги:  
сначала считается $QK^T$, затем деление на $\sqrt{d_k}$, потом SoftMax, после чего матрица весов умножается на $V$, чтобы получить context vectors.

## 5.3. Зачем нужно деление на $\sqrt{d_k}$

Если размерность векторов велика, скалярные произведения растут по масштабу, и SoftMax может становиться слишком “резким”. Деление на $\sqrt{d_k}$ стабилизирует масштаб logits и облегчает обучение.

## 5.4. Алгоритм

1. Вычислить similarity matrix:
$$
S = QK^T
$$

2. Отмасштабировать:
$$
\tilde S = \frac{S}{\sqrt{d_k}}
$$

3. Применить SoftMax по keys:
$$
A = \text{softmax}(\tilde S)
$$

4. Получить контекст:
$$
C = AV
$$

## 5.5. Псевдокод

```text
function scaled_dot_product_attention(Q, K, V):
    scores = (Q @ K.T) / sqrt(d_k)
    weights = softmax(scores)
    context = weights @ V
    return context
```

---

# 6. Виды attention в transformers

Курс явно выделяет три основных варианта attention в transformer model.

## 6.1. Encoder-Decoder Attention

### Интуиция
Слова одной последовательности attend к словам другой последовательности.

### Формально
- queries приходят из decoder states;
- keys и values приходят из encoder outputs.

### Применение
- machine translation;
- summarization;
- dialogue generation.

## 6.2. Self-Attention

### Интуиция
Каждый токен смотрит на все токены той же последовательности.

### Формально
queries, keys и values происходят из одного и того же input sequence.

### Смысл
Self-attention даёт **contextual representation** каждого токена, учитывающую всю последовательность. Курс прямо формулирует это как “representation of the meaning of each word within the sentence”.

## 6.3. Masked Self-Attention

### Интуиция
В генерации токен не должен “подглядывать” в будущее.

### Формально
queries, keys и values тоже приходят из одной последовательности, но для позиции $t$ запрещены attention weights на позиции $> t$.

### Где используется
- transformer decoder;
- GPT-like models;
- autoregressive language modeling.

Курс прямо указывает, что masked self-attention присутствует в decoder и гарантирует зависимость только от already known outputs.

---

# 7. Masked Self-Attention

## 7.1. Формальная постановка

Обычное self-attention:

$$
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

Для masked self-attention добавляется маска $M$:

$$
\text{MaskedAttention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$

где:

$$
M_{ij} =
\begin{cases}
0, & j \le i \\
-\infty, & j > i
\end{cases}
$$

На практике вместо $-\infty$ берут очень большое отрицательное число.

Курс описывает именно такую маску: нули на разрешённых позициях и minus infinity above the diagonal, чтобы после SoftMax веса на будущих позициях стали нулевыми.

## 7.2. Зачем это нужно

Без masking decoder при обучении мог бы видеть правильные будущие слова и задача next-token prediction стала бы некорректной.

## 7.3. Псевдокод

```text
function masked_self_attention(Q, K, V, mask):
    scores = (Q @ K.T) / sqrt(d_k)
    scores = scores + mask
    weights = softmax(scores)
    return weights @ V
```

---

# 8. Multi-Head Attention

## 8.1. Интуитивная идея

Один attention head — это один способ смотреть на взаимосвязи между токенами. Но язык многослоен: одни зависимости синтаксические, другие семантические, третьи локальные, четвёртые дальние.

Поэтому transformer использует **несколько attention heads параллельно**.

Курс объясняет multi-head attention как набор параллельных scaled dot-product attention computations после разных линейных преобразований Q, K и V, а затем конкатенацию результатов и финальную линейную проекцию.

## 8.2. Формальная постановка

Для каждой головы $h$:

$$
Q_h = QW_h^Q,\quad K_h = KW_h^K,\quad V_h = VW_h^V
$$

Далее:

$$
\text{head}_h = \text{Attention}(Q_h, K_h, V_h)
$$

После этого:

$$
\text{MultiHead}(Q,K,V)
=
\text{Concat}(\text{head}_1,\dots,\text{head}_H)W^O
$$

## 8.3. Размерности

Курс отдельно разбирает размерности:

- входные Q/K/V имеют ширину $d_{\text{model}}$;
- для каждой головы используются отдельные матрицы $W^Q, W^K, W^V$;
- в оригинальном transformer обычно рекомендуют $d_k=d_v=d_{\text{model}}/H$, где $H$ — число голов.

## 8.4. Почему это работает

Разные heads могут учиться разным типам отношений:

- согласование подлежащего и сказуемого;
- coreference;
- local phrase structure;
- long-range dependencies;
- task-specific salience.

## 8.5. Псевдокод

```text
function multi_head_attention(Q, K, V):
    heads = []
    for h in 1..H:
        Q_h = Q @ WQ[h]
        K_h = K @ WK[h]
        V_h = V @ WV[h]
        heads.append(attention(Q_h, K_h, V_h))
    H_cat = concat(heads)
    return H_cat @ WO
```

---

# 9. Transformer Architecture

## 9.1. Интуитивная идея

Transformer заменяет рекуррентное прохождение по последовательности на стек attention-блоков и position-wise feed-forward blocks. За счёт этого модель:

- видит широкий контекст;
- хорошо параллелится;
- масштабируется на большие данные и вычислительные ресурсы.

Курс прямо подчёркивает, что transformer architecture легко parallelize по сравнению с RNN models и потому обучается существенно эффективнее на multiple GPUs.

## 9.2. Общая структура

Оригинальный transformer состоит из:

- **encoder stack**;
- **decoder stack**.

Каждый encoder layer содержит:

1. multi-head self-attention;
2. residual connection + normalization;
3. feed-forward layer;
4. residual connection + normalization.

Каждый decoder layer содержит:

1. masked multi-head self-attention;
2. residual + normalization;
3. encoder-decoder attention;
4. residual + normalization;
5. feed-forward;
6. residual + normalization.

Именно такую структуру курс кратко описывает в обзорном видео по transformer.

## 9.3. Encoder

### Что делает
Строит contextual representations входных токенов.

Курс говорит об этом буквально: благодаря self-attention encoder даёт contextual representation of each one of your inputs.

## 9.4. Decoder

### Что делает
Генерирует выход слева направо, используя:

- прошлые сгенерированные токены;
- encoder outputs.

В decoder первый attention-блок masked, а второй обращается к encoder outputs. Курс подчёркивает именно такую двухступенчатую структуру.

---

# 10. Positional Encoding

## 10.1. Почему оно нужно

Transformer не использует рекуррентность, а значит сам по себе не знает порядок токенов. Но порядок слов критичен для языка.

Курс прямо говорит, что positional encoding необходим, потому что transformers don’t use recurrent neural networks, but word order is relevant for any language.

## 10.2. Идея

К embeddings токенов добавляется вектор, кодирующий позицию:

$$
z_t = e_t + p_t
$$

где:

- $e_t$ — token embedding;
- $p_t$ — positional encoding.

Курс отмечает, что positional encoding может быть learned или fixed.

## 10.3. Варианты

### Fixed sinusoidal encoding
Классический вариант из оригинального transformer:

$$
PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

### Learned positional embeddings
Модель учит отдельный embedding для каждой позиции. Курс приводит пример decoder, где к word embeddings добавляются learned vectors for positions 1, 2, 3, … 

---

# 11. Transformer Decoder и GPT-style models

## 11.1. Интуитивная идея

Если взять только decoder stack с masked self-attention, получится архитектура для autoregressive text generation.

Курс прямо проводит такую линию: GPT использует decoder stacks only, а в построении decoder model токенизированное предложение проходит через word embeddings, затем к ним добавляются position vectors, затем идут multi-head attention и feed-forward blocks.

## 11.2. Формально

Decoder-only LM моделирует:

$$
P(x_1,\dots,x_T)=\prod_{t=1}^{T}P(x_t \mid x_{<t})
$$

Это causal language modeling.

## 11.3. Почему decoder-only architecture так важна

Она естественно подходит для:

- next-token prediction;
- dialogue;
- code generation;
- long-form text generation;
- instruction following.

Именно из этой линии выросли GPT, LLaMA, Mistral и другие современные LLM.

---

# 12. Transformer Encoder и BERT-style models

## 12.1. Интуитивная идея

Если взять только encoder stack, получится модель, ориентированная не на генерацию слева направо, а на **двустороннее понимание контекста**.

Курс прямо противопоставляет GPT и BERT: GPT — decoder-only, BERT — encoder-only, и подчёркивает, что BERT использует bidirectional encoder representations from transformers, что позволяет смотреть и слева, и справа при предсказании скрытых слов.

## 12.2. Pretraining objective: Masked Language Modeling

В BERT часть токенов маскируется, и модель должна восстановить их по двустороннему контексту:

$$
P(x_m \mid x_{\setminus m})
$$

где $x_m$ — masked token.

Курс говорит, что BERT pre-training uses masked language modeling and next sentence prediction.

## 12.3. Next Sentence Prediction

BERT в базовой версии также обучался предсказывать, следует ли sentence B за sentence A.

Это использовалось как дополнительный pretraining signal для межфразовых отношений. Курс описывает именно такую постановку: given sentence A, predict whether sentence B actually follows it. 

## 12.4. Где BERT особенно силён

- classification;
- token classification;
- NER;
- sentence pair tasks;
- extractive QA;
- semantic similarity.

---

# 13. Encoder-Decoder Transformers: T5 и подобные модели

## 13.1. Интуитивная идея

Если сохранить обе части оригинального transformer — encoder и decoder, — получаем очень гибкую архитектуру для text-to-text tasks.

Курс отмечает, что T5 tested the original encoder-decoder setup and researchers found better performance when both encoder and decoder stacks are present. 

## 13.2. Text-to-Text парадигма

T5 формулирует множество задач как преобразование строки в строку:

- translation;
- classification;
- question answering;
- summarization;
- regression-like scoring.

Курс подробно показывает, как одна и та же T5 model может выполнять разные задачи, если в input string добавить текстовую инструкцию задачи, например перевод, question answering или sentence acceptability. 

## 13.3. Почему это важно

Эта идея стала прямым предшественником современных instruction-tuned models.  
Смысл в том, что одна архитектура может быть унифицирована под множество задач без создания отдельной модели на каждую.

---

# 14. Encoder-only vs Decoder-only vs Encoder-Decoder

Это один из главных концептуальных итогов attention era.

## 14.1. Encoder-only
### Пример
BERT, RoBERTa, DeBERTa.

### Лучшие задачи
- understanding;
- classification;
- token labeling;
- retrieval encoders.

### Сильная сторона
Bidirectional contextual representations.

## 14.2. Decoder-only
### Пример
GPT, LLaMA, Mistral.

### Лучшие задачи
- generation;
- dialogue;
- completion;
- code generation.

### Сильная сторона
Autoregressive next-token prediction.

## 14.3. Encoder-Decoder
### Пример
T5, BART.

### Лучшие задачи
- translation;
- summarization;
- structured generation;
- task-conditioned output.

### Сильная сторона
Явное разделение encoding input и decoding output.

Курс прямо делает этот разрез: transformer = encoder + decoder, GPT = decoder stack only, BERT = encoder stack only, T5 = encoder-decoder again.

---

# 15. Tokenization в современных attention models

## 15.1. Базовая идея

Любая transformer model работает не с “сырыми словами”, а с дискретными токенами, которые затем превращаются в integer ids и embeddings.

В субтитрах курса на примере machine translation показано, что предложения сначала токенизируются, затем переводятся в indices через word-to-index mapping, добавляются EOS token и padding zeros до нужной длины.

## 15.2. Почему word-level tokenization недостаточна

Полнословная токенизация создаёт проблемы:

- огромный vocabulary;
- OOV words;
- слабая работа с морфологией;
- плохая переносимость между доменами.

Поэтому современные attention models почти всегда используют **subword tokenization**.

## 15.3. BPE, WordPiece, SentencePiece

Хотя в данных субтитрах эти методы подробно не развёрнуты, в современной системе знаний по NLP их обязательно нужно включить.

### BPE
Итеративно сливает самые частые пары символов/подслов.

### WordPiece
Похожа по духу на BPE, но оптимизирует выбор merge’ов по вероятностному критерию; исторически связана с BERT-like tokenizers.

### SentencePiece
Работает без жёсткой предварительной токенизации по пробелам и удобна для multilingual setups.

## 15.4. Почему subword tokenization стала стандартом

Она позволяет:

- уменьшить vocabulary size;
- разбирать редкие слова на известные части;
- лучше работать с морфологией;
- снизить OOV problem;
- обучать multilingual and large-scale models.

## 15.5. Формально

Пусть исходная строка $s$ токенизируется в последовательность subword units:

$$
s \mapsto (t_1,\dots,t_n)
$$

Далее:

$$
t_i \mapsto id_i \mapsto e_i
$$

И уже embeddings $e_i$ поступают в transformer.

---

# 16. Pretraining and Transfer Learning

## 16.1. Главная идея

Attention models стали особенно мощными не только из-за архитектуры, но и из-за режима обучения:

1. сначала large-scale pretraining;
2. потом fine-tuning или instruction tuning под конкретные задачи.

Курс на примерах BERT, GPT и T5 как раз показывает эту развилку: разные архитектуры + разные pretraining objectives → разные сильные стороны на downstream tasks.

## 16.2. Основные pretraining objectives

### Masked Language Modeling
Используется в BERT:
$$
P(x_m \mid x_{\setminus m})
$$

### Causal Language Modeling
Используется в GPT:
$$
P(x_t \mid x_{<t})
$$

### Sequence-to-Sequence Denoising / Text-to-Text
Используется в T5-подобных моделях:
input corruption or task prefix → output sequence.

## 16.3. Fine-tuning

После pretraining модель дообучается под конкретную задачу:

- classification;
- NER;
- QA;
- summarization;
- translation.

## 16.4. Почему transfer learning изменил NLP

Раньше модель часто обучали под одну задачу с нуля.  
Теперь большая часть языковых знаний извлекается на этапе pretraining, а downstream adaptation требует гораздо меньше task-specific data.

---

# 17. Почему transformers вытеснили RNN

Курс даёт на это прямой ответ:  
RNN страдают из-за своей sequential structure, плохо используют parallel computing и испытывают трудности на длинных контекстах; transformers благодаря attention лучше параллелятся и масштабируются.

## 17.1. Преимущества transformers

### 1. Глобальный контекст
Каждый токен может attend ко всем другим токенам уже в одном слое.

### 2. Параллелизация
Нет рекуррентной зависимости вида $h_t \leftarrow h_{t-1}$ при вычислении всех hidden states.

### 3. Масштабируемость
Архитектура хорошо подходит для больших данных и крупных вычислительных кластеров.

### 4. Гибкость
Одна и та же базовая идея работает и для:
- understanding;
- generation;
- translation;
- multitask learning.

## 17.2. Недостатки transformers

Нужно честно отметить и ограничения:

- attention cost растёт квадратично по длине последовательности;
- большие модели дороги в обучении и inference;
- требуют большого количества данных и вычислений;
- нуждаются в careful tokenization and optimization.

Именно поэтому в современном NLP активно развиваются efficient attention variants, retrieval augmentation, quantization и long-context methods.

---

# 18. Практические NLP-задачи в attention era

## 18.1. Machine Translation
Encoder-decoder transformer.  
Курс выводит attention из translation setup и показывает, как alignment помогает работать с разным порядком слов.

## 18.2. Question Answering
- BERT-like models: extractive QA;
- T5/GPT-like models: generative QA.

Курс использует QA как один из демонстрационных multitask examples for T5.

## 18.3. Classification
Encoder-only или T5-style instruction prompt.  
В курсе есть пример, где T5 по текстовому префиксу выполняет sentence classification.

## 18.4. Summarization
Обычно encoder-decoder or decoder-only generation.

## 18.5. Text Generation
Decoder-only transformers, causal masking, next-token prediction.

## 18.6. Semantic Search / Retrieval
Encoder representations или dual-encoder setups, которые затем могут использоваться с vector search systems.

---

# 19. Практический pipeline современного attention-based NLP

```text
raw text
→ tokenizer / subword tokenizer
→ token ids
→ embeddings + positional encoding
→ transformer layers
→ task-specific head or decoder generation
→ fine-tuning / inference
```

### Для BERT-like модели
```text
text
→ WordPiece/subword tokens
→ encoder stack
→ [CLS] or token representations
→ classifier / tagger / QA head
```

### Для GPT-like модели
```text
prompt
→ subword tokens
→ decoder-only transformer with masked self-attention
→ next-token distribution
→ autoregressive decoding
```

### Для T5-like модели
```text
task prefix + input text
→ encoder
→ decoder
→ text output
```

---

# 20. Короткие Python-примеры

## 20.1. Scaled Dot-Product Attention на PyTorch

```python
import torch
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights
```

## 20.2. Multi-Head Attention sketch

```python
import torch
import torch.nn as nn

class SimpleMultiHeadAttention(nn.Module):
    def __init__(self, d_model=128, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        Q = self.Wq(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.Wk(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.Wv(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        out = weights @ V

        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.Wo(out)
```

## 20.3. Causal mask

```python
import torch

def causal_mask(seq_len: int):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask
```

## 20.4. Hugging Face: BERT classification

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

text = "Transformers changed NLP."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
logits = outputs.logits
```

## 20.5. Hugging Face: GPT generation

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Natural language processing is"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## 20.6. Hugging Face: T5 text-to-text

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = "translate English to French: I am happy"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

---

# 21. Что нужно усвоить по этому разделу

После этого раздела ты должен уметь:

1. объяснить, почему attention решает bottleneck seq2seq;
2. различать queries, keys и values;
3. записать формулу scaled dot-product attention;
4. объяснить self-attention и masked self-attention;
5. понимать, зачем нужен multi-head attention;
6. описать структуру transformer encoder и decoder;
7. объяснить роль positional encoding;
8. различать encoder-only, decoder-only и encoder-decoder transformers;
9. понимать разницу между MLM и causal LM;
10. объяснить, почему BERT подходит для understanding, а GPT — для generation;
11. понимать, как T5 превращает многие задачи в text-to-text;
12. видеть, как из transformer architecture выросли современные LLM.

---

# 22. Итог раздела

Attention models стали тем моментом, когда NLP окончательно перешёл от “локального” понимания текста к **глобальному контекстному моделированию**.

Главный смысл этого этапа:

### Было
- короткий контекст в n-grams;
- скрытые состояния в RNN/LSTM;
- information bottleneck в seq2seq.

### Стало
- динамический выбор релевантных частей контекста;
- self-attention для каждого токена;
- архитектуры без рекуррентности;
- масштабируемые transformers;
- pretraining at scale;
- foundation for modern LLMs.

Именно attention models сделали возможным современный NLP в его нынешнем виде:

- BERT для понимания текста;
- GPT для генерации;
- T5/BART для text-to-text задач;
- instruction tuning, retrieval augmentation и LLM-based systems как следующий логический слой над transformer foundation.